# 03 &middot; Structural analysis

**This is the notebook that answers Reviewer 3's central objection**, that the
paper's findings did not require process mining.

Computes, on the full log rather than a sample:

* self-loop burden across all directly-follows transitions
* per-activity self-loop rates
* quiz-attempt cycle statistics (views per attempt, duration, inter-view gap)
* A&rarr;B&rarr;A rework patterns
* transition latencies
* **feedback latency** (submission &rarr; grading), if lecturer data is present

### Measurement caveat

`objectid` is dropped during preprocessing, so the deposit contains no native
quiz-attempt key. Attempts are segmented here by treating each `attempt_started`
within a course-level case as a boundary. Retain `objectid` in preprocessing and
re-run before these figures inform operational decisions.

## 1. Setup

Run this section first. It installs dependencies and downloads the deposit from
figshare into the Colab VM.

**Runtime:** Runtime &rarr; Change runtime type &rarr; **High-RAM** if available.
The largest faculty file (FIF, 1.3 GB on disk) needs roughly 6 GB once loaded.

In [ ]:
#@title Install dependencies { display-mode: "form" }
!pip install -q pm4py==2.7.23.3 statsmodels 2>/dev/null
import os
os.environ["TQDM_DISABLE"] = "1"

import warnings, sys, json, time, gc, random
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np

print("Python ", sys.version.split()[0])
print("pandas ", pd.__version__)
import pm4py; print("pm4py  ", pm4py.__version__)

# --- RAM report -------------------------------------------------------------
try:
    import psutil
    gb = psutil.virtual_memory().total / 1e9
    print(f"RAM    {gb:.1f} GB")
    if gb < 20:
        print("\\n  NOTE: standard runtime. FIF and FTE may run out of memory.")
        print("  Runtime -> Change runtime type -> High-RAM is recommended.")
except Exception:
    pass

In [ ]:
#@title Download the deposit from figshare { display-mode: "form" }
# Queries the figshare API, so lecturer files are picked up automatically
# once they are added to the deposit.

import requests, os, pathlib

ARTICLE = "28341992"          #@param {type:"string"}
DATA_DIR = "/content/data"    #@param {type:"string"}
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

meta = requests.get(f"https://api.figshare.com/v2/articles/{ARTICLE}", timeout=60).json()
print(f"{meta['title']}  (v{meta.get('version','?')})")
print(f"{len(meta['files'])} files, {meta['size']/1e9:.2f} GB total\n")

FILES = {}
for f in meta["files"]:
    FILES[f["name"]] = f["download_url"]
    print(f"  {f['name']:<32} {f['size']/1e6:>8.1f} MB")

# --- completeness check -----------------------------------------------------
FACULTIES = ["FEB", "FIF", "FIK", "FIT", "FKB", "FRI", "FTE"]
missing = [f"{fac}_{role}.csv" for fac in FACULTIES
           for role in ("Student", "Lecturer") if f"{fac}_{role}.csv" not in FILES]
if missing:
    print("\n  MISSING FROM DEPOSIT:")
    for m in missing:
        print(f"    {m}")
    print("\n  Analyses for these partitions will be skipped.")


def fetch(name):
    """Download one file if not already present. Returns local path or None."""
    if name not in FILES:
        return None
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        return dest
    print(f"downloading {name} ...", flush=True)
    with requests.get(FILES[name], stream=True, timeout=1800) as r:
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 22):
                fh.write(chunk)
    print(f"  -> {os.path.getsize(dest)/1e6:.0f} MB")
    return dest

In [ ]:
#@title Loader { display-mode: "form" }
# Memory-efficient reader plus the two corrections identified during revision.

USECOLS = ["id", "eventname", "component", "action", "target", "crud",
           "edulevel", "userid", "courseid", "timecreated", "event"]
DTYPES = {"id": "int64", "eventname": "category", "component": "category",
          "action": "category", "target": "category", "crud": "category",
          "edulevel": "int8", "userid": "int32", "courseid": "int32",
          "event": "category"}

CUTOFF = "2023-06-26"   # verified coverage boundary (NOT July, see manuscript)


def load(faculty, role, apply_dedup=True, cols=None):
    """Load one faculty-role partition.

    apply_dedup fixes the defect found during revision: the original notebooks
    called df.drop_duplicates() WITHOUT assignment, so duplicates were counted
    and reported but never removed from the working data.
    """
    name = f"{faculty}_{role}.csv"
    path = fetch(name)
    if path is None:
        print(f"  [skip] {name} not in deposit")
        return None
    use = cols or USECOLS
    df = pd.read_csv(path, index_col=0, usecols=lambda c: c in use or c == "Unnamed: 0",
                     dtype={k: v for k, v in DTYPES.items() if k in use},
                     parse_dates=["timecreated"] if "timecreated" in use else None)
    n_raw = len(df)
    n_dup = int(df.duplicated().sum())
    if apply_dedup and n_dup:
        df = df.drop_duplicates()          # assignment: this is the fix
    df.attrs["n_raw"] = n_raw
    df.attrs["n_dup"] = n_dup
    df.attrs["faculty"] = faculty
    df.attrs["role"] = role
    return df


def add_case(df, notion):
    """notion is 'user' or 'course'."""
    if notion == "user":
        df["case"] = df["userid"].astype(str)
    else:
        df["case"] = df["userid"].astype(str) + "_" + df["courseid"].astype(str)
    return df


def to_log(df, notion, sample=None, seed=42, max_len=None):
    """Build a pm4py EventLog. sample caps the number of traces."""
    d = add_case(df, notion)
    if max_len:
        keep = d.groupby("case").size()
        d = d[d["case"].isin(keep[keep <= max_len].index)]
    if sample:
        random.seed(seed)
        cases = sorted(d["case"].unique())
        d = d[d["case"].isin(set(random.sample(cases, min(sample, len(cases)))))]
    ldf = (d[["case", "event", "timecreated"]]
           .rename(columns={"case": "case:concept:name", "event": "concept:name",
                            "timecreated": "time:timestamp"})
           .sort_values(["case:concept:name", "time:timestamp"])
           .reset_index(drop=True))
    # pm4py rejects categorical columns: cast the two key columns to str
    ldf["case:concept:name"] = ldf["case:concept:name"].astype(str)
    ldf["concept:name"] = ldf["concept:name"].astype(str)
    return pm4py.convert_to_event_log(ldf), ldf


def save(obj, name):
    """Persist a result table and offer it for download."""
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(f"/content/{name}.csv", index=False)
    else:
        json.dump(obj, open(f"/content/{name}.json", "w"), indent=1)
    print(f"saved /content/{name}")

## 2. Directly-follows structure

In [ ]:
#@title { display-mode: "form" }
RUN_FACULTIES = ["FIT", "FKB", "FIK", "FRI", "FEB", "FTE", "FIF"]  #@param
NOTION = "course"  #@param ["course", "user"]

struct_rows = []
dfg_store = {}

for fac in RUN_FACULTIES:
    df = load(fac, "Student")
    if df is None:
        continue
    df = add_case(df, NOTION).sort_values(["case", "timecreated"]).reset_index(drop=True)
    df["next"] = df["event"].shift(-1)
    df["next_case"] = df["case"].shift(-1)
    df["dt"] = (df["timecreated"].shift(-1) - df["timecreated"]).dt.total_seconds()
    same = df["case"] == df["next_case"]
    dfg = df[same]

    sl = dfg[dfg["event"] == dfg["next"]]
    edges = dfg.groupby(["event", "next"], observed=True).size().sort_values(ascending=False)
    top = edges.index[0]

    struct_rows.append(dict(
        faculty=fac, transitions=len(dfg),
        self_loops=len(sl), pct_self_loop=round(100 * len(sl) / len(dfg), 1),
        top_edge_from=top[0], top_edge_to=top[1],
        top_edge_n=int(edges.iloc[0]),
        pct_top_edge=round(100 * edges.iloc[0] / len(dfg), 1),
    ))
    dfg_store[fac] = edges.head(40)

    print(f"\n=== {fac} ===")
    print(f"  self-loops: {len(sl):,}/{len(dfg):,} = {100*len(sl)/len(dfg):.1f}% "
          f"of all directly-follows transitions")
    print(f"  top edge  : {top[0]} -> {top[1]}  "
          f"{edges.iloc[0]:,} ({100*edges.iloc[0]/len(dfg):.1f}%)")
    print("  top self-looping activities:")
    for a, n in sl.groupby("event", observed=True).size().sort_values(
            ascending=False).head(5).items():
        base = int((df["event"] == a).sum())
        print(f"    {str(a)[:52]:<52} {n:>9,}  ({100*n/base:.1f}% of its events)")
    del df, dfg, sl; gc.collect()

structure = pd.DataFrame(struct_rows)
save(structure, "03_structure")
structure

## 3. Quiz-attempt cycle

The finding: the most frequent student activity is one page re-rendering, not
distinct engagement.

In [ ]:
QUIZ = ["mod_quiz\\attempt_started", "mod_quiz\\attempt_viewed",
        "mod_quiz\\attempt_submitted", "mod_quiz\\attempt_summary_viewed",
        "mod_quiz\\attempt_reviewed", "mod_quiz\\attempt_abandoned"]
quiz_rows = []

for fac in RUN_FACULTIES:
    df = load(fac, "Student", cols=["event", "userid", "courseid", "timecreated"])
    if df is None:
        continue
    df = add_case(df, "course")
    q = df[df["event"].astype(str).isin(QUIZ)].copy()
    if not len(q):
        print(f"[{fac}] no quiz events"); del df; gc.collect(); continue

    # Attempts are segmented by attempt_started boundaries within a course-level
    # case because objectid / the native attempt key is not present in the
    # deposited processed files.
    q = q.sort_values(["case", "timecreated"])
    q["is_start"] = (q["event"].astype(str) == "mod_quiz\\attempt_started").astype(int)
    q["attempt_no"] = q.groupby("case")["is_start"].cumsum()
    q = q[q["attempt_no"] > 0]

    g = q.groupby(["case", "attempt_no"])
    views = g["event"].apply(
        lambda x: (x.astype(str) == "mod_quiz\\attempt_viewed").sum())
    sub = g["event"].apply(
        lambda x: (x.astype(str) == "mod_quiz\\attempt_submitted").any())
    dur = g["timecreated"].agg(
        lambda s: (s.max() - s.min()).total_seconds() / 60)

    # Reproduce the reported short re-rendering interval. The gap is computed
    # between consecutive attempt_viewed events within the SAME segmented
    # attempt; the first view of each attempt has no preceding gap.
    qv = q[q["event"].astype(str) == "mod_quiz\\attempt_viewed"][
        ["case", "attempt_no", "timecreated"]].copy()
    qv["prev_view"] = qv.groupby(["case", "attempt_no"])["timecreated"].shift(1)
    qv["inter_view_s"] = (
        qv["timecreated"] - qv["prev_view"]).dt.total_seconds()
    gaps = qv["inter_view_s"].dropna()
    # Zero-second gaps can be legitimate multiple log writes; retain them
    # because they are part of the recorded platform behaviour.

    hi = views[views >= 10]
    quiz_rows.append(dict(
        faculty=fac,
        attempts=len(views),
        views_mean=round(float(views.mean()), 1),
        views_median=int(views.median()),
        views_p90=int(views.quantile(.9)),
        views_max=int(views.max()),
        pct_submitted=round(100 * float(sub.mean()), 1),
        dur_median_min=round(float(dur.median()), 1),
        dur_p90_min=round(float(dur.quantile(.9)), 1),
        inter_view_n=int(len(gaps)),
        inter_view_median_s=round(float(gaps.median()), 1) if len(gaps) else np.nan,
        inter_view_p90_s=round(float(gaps.quantile(.9)), 1) if len(gaps) else np.nan,
        pct_attempts_10plus=round(100 * len(hi) / len(views), 1),
        pct_views_in_those=round(100 * hi.sum() / views.sum(), 1),
    ))

    gap_txt = (f"{float(gaps.median()):.1f} s median inter-view gap"
               if len(gaps) else "no repeated-view gaps")
    print(f"{fac}: {len(views):,} attempts | median {int(views.median())} views "
          f"| {float(dur.median()):.1f} min | {gap_txt} "
          f"| {100*float(sub.mean()):.1f}% submitted "
          f"| {100*len(hi)/len(views):.1f}% have 10+ revisits", flush=True)
    del df, q, qv, g; gc.collect()

quizstats = pd.DataFrame(quiz_rows)
save(quizstats, "03_quiz_cycle")
quizstats


## 4. Feedback latency (submission &rarr; grading)

The bottleneck metric flagged in the manuscript as the strongest remaining
candidate. **Requires lecturer partitions in the deposit.** It joins student
submissions to lecturer grading events on `courseid`, so it estimates
course-level turnaround rather than per-submission latency.

In [ ]:
lat_rows = []
for fac in RUN_FACULTIES:
    stu = load(fac, "Student", cols=["event", "userid", "courseid", "timecreated"])
    lec = load(fac, "Lecturer", cols=["event", "userid", "courseid", "timecreated"])
    if stu is None or lec is None:
        print(f"[{fac}] needs both partitions - skipped")
        continue
    subs = stu[stu["event"].astype(str).str.contains("assessable_submitted", na=False)]
    grades = lec[lec["event"].astype(str).str.contains("user_graded", na=False)]
    if not len(subs) or not len(grades):
        print(f"[{fac}] no submission/grading events"); continue

    gmed = grades.groupby("courseid")["timecreated"].apply(list)
    out = []
    for cid, t in subs.groupby("courseid")["timecreated"]:
        if cid not in gmed:
            continue
        gt = np.sort(np.array(gmed[cid], dtype="datetime64[ns]"))
        for ts in t.values:
            i = np.searchsorted(gt, ts)
            if i < len(gt):
                out.append((gt[i] - ts) / np.timedelta64(1, "h"))
    if out:
        out = np.array(out)
        lat_rows.append(dict(faculty=fac, n=len(out),
                             median_h=round(float(np.median(out)), 1),
                             p90_h=round(float(np.percentile(out, 90)), 1),
                             median_days=round(float(np.median(out)) / 24, 1)))
        print(f"{fac}: n={len(out):,} median {np.median(out):.1f} h "
              f"({np.median(out)/24:.1f} days), p90 {np.percentile(out,90):.1f} h")
    del stu, lec; gc.collect()

if lat_rows:
    latency = pd.DataFrame(lat_rows)
    save(latency, "03_feedback_latency")
    display(latency)
else:
    print("\nNo latency computed. Upload the lecturer CSVs to the figshare deposit.")

## 5. Rework patterns

In [ ]:
rework_rows = []
for fac in RUN_FACULTIES:
    df = load(fac, "Student", cols=["event", "userid", "courseid", "timecreated"])
    if df is None:
        continue
    df = add_case(df, "course").sort_values(
        ["case", "timecreated"]).reset_index(drop=True)
    d = df.copy()
    d["n1"] = d["event"].shift(-1); d["c1"] = d["case"].shift(-1)
    d["n2"] = d["event"].shift(-2); d["c2"] = d["case"].shift(-2)
    d = d[(d["case"] == d["c1"]) & (d["case"] == d["c2"])
          & (d["event"].astype(str) == d["n2"].astype(str))
          & (d["event"].astype(str) != d["n1"].astype(str))]

    counts = (d.groupby(["event", "n1"], observed=True).size()
              .sort_values(ascending=False))
    print(f"\n=== {fac}: top A->B->A rework ===")
    for (a, b), n in counts.head(10).items():
        rework_rows.append(dict(faculty=fac, activity_a=str(a),
                                activity_b=str(b), occurrences=int(n)))
        print(f"  {str(a)[:38]:<38} -> {str(b)[:38]:<38} -> back  {n:>8,}")
    del df, d; gc.collect()

rework = pd.DataFrame(rework_rows)
save(rework, "03_rework_patterns")
rework
